<a href="https://colab.research.google.com/github/shahedtheboss/SECOM_kaggle_comp/blob/main/SECOM_kaggle_comp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install -q pyarrow

In [22]:
import pandas as pd
import numpy as np
import kagglehub
from pathlib import Path
import pyarrow.parquet as pq
from sklearn.model_selection import train_test_split,StratifiedKFold, RandomizedSearchCV,cross_val_predict,cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report,recall_score,precision_score,f1_score,accuracy_score
from sklearn.pipeline import Pipeline


In [3]:
kagglehub.login()


Kaggle credentials set.
Kaggle credentials successfully validated.


In [3]:
path = kagglehub.competition_download(
    'secom-defect-detection-challenge'
)

print(path)

/root/.cache/kagglehub/competitions/secom-defect-detection-challenge


In [4]:
competition_dir = Path(path)


for item in competition_dir.iterdir():
  print(item)

/root/.cache/kagglehub/competitions/secom-defect-detection-challenge/sample_submission.csv
/root/.cache/kagglehub/competitions/secom-defect-detection-challenge/test.csv
/root/.cache/kagglehub/competitions/secom-defect-detection-challenge/train.csv


In [6]:
train = pd.read_csv(competition_dir / 'train.csv')
print("Training data loaded!")
test = pd.read_csv(competition_dir / 'test.csv')
print("Test data loaded!")

sample_submission = pd.read_csv(competition_dir / 'sample_submission.csv')
print("Sample Data loaded!")

Training data loaded!
Test data loaded!
Sample Data loaded!


In [7]:
print("Train Shape:", train.shape)
print("Test Shape:", test.shape)
print("Sample Shape:", sample_submission.shape)
# print("Training Schema:")
# print(train.schema)

# print("Test Shape:", test.shape)
# print("Sample Submission Shape:", sample_submisssion.shape)

Train Shape: (1253, 593)
Test Shape: (314, 592)
Sample Shape: (314, 2)


In [ ]:

print("TRAIN COLUMNS:")
print(train.columns.tolist())

print("\nTEST COLUMNS:")
print(test.columns.tolist())

print("\nSAMPLE SUBMISSION COLUMNS:")
print(sample_submission.columns.tolist())

TRAIN COLUMNS:
['id', 'time', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99', '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '110', '111', '112', '113', '114', '115', '116', '117', '118', '119', '120', '121', '122', '123', '124', '125', '126', '127', '128', '129', '130', '131', '132', '133', '134', '135', '136', '137', '138', '139', '140', '141', '142', '143', '144', '145', '146', '147', '148', '149', '150', '151', '152', '153', '1

In [ ]:
display(train.head())


,id,time,0,1,2,3,4,5,6,7,...,581,582,583,584,585,586,587,588,589,target
0,1198,2008-09-30 15:26:00,3075.32,2491.07,2185.1000,1201.0491,0.7821,100.0,105.8489,0.1208,...,NaN,0.4974,0.0128,0.0033,2.5767,0.0223,0.0105,0.0034,47.0690,-1
1,436,2008-08-23 05:58:00,3071.58,2489.47,2217.3777,1425.1041,1.7585,100.0,106.2556,0.1200,...,16.6695,0.5004,0.0316,0.0066,6.3183,0.0329,0.0055,0.0022,16.6695,-1
2,635,2008-01-09 20:51:00,3017.53,2524.09,2201.0667,880.2317,1.4148,100.0,106.5478,0.1211,...,NaN,0.4998,0.0097,0.0026,1.9495,0.0328,0.0235,0.0068,71.5333,-1
3,996,2008-09-21 19:19:00,2901.62,2569.45,2223.9000,1745.3724,1.9974,100.0,96.7567,0.1241,...,NaN,0.5004,0.0174,0.0034,3.4771,0.0200,0.0205,0.0061,102.5241,-1
4,782,2008-11-09 07:43:00,2982.59,2466.86,2117.5889,894.0996,1.4330,100.0,106.4944,0.1253,...,NaN,0.4986,0.0172,0.0038,3.4561,0.0373,0.0079,0.0030,21.0599,-1


In [ ]:
# Data types and non-null counts
print("TRAIN INFO:")
train.info()

print("\nTEST INFO:")
test.info()

TRAIN INFO:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1253 entries, 0 to 1252
Columns: 593 entries, id to target
dtypes: float64(590), int64(2), object(1)
memory usage: 5.7+ MB

TEST INFO:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 314 entries, 0 to 313
Columns: 592 entries, id to 589
dtypes: float64(590), int64(1), object(1)
memory usage: 1.4+ MB


In [ ]:
print("Train missing values:")
print(train.isnull().sum())

print("\nTest missing values:")
print(test.isnull().sum())

Train missing values:
id         0
time       0
0          5
1          5
2         13
          ..
586        1
587        1
588        1
589        1
target     0
Length: 593, dtype: int64

Test missing values:
id      0
time    0
0       1
1       2
2       1
       ..
585     0
586     0
587     0
588     0
589     0
Length: 592, dtype: int64


In [ ]:
display(train.head())

display(test.head())

print("Data types")
print(train.dtypes.value_counts())

print("Target data types:")
print(train["target"].dtype)

print("Number of unique target values:")
print(train["target"].nunique())

print("First 20 target values:")
print(train["target"].head(20).tolist())

,id,time,0,1,2,3,4,5,6,7,...,581,582,583,584,585,586,587,588,589,target
0,1198,2008-09-30 15:26:00,3075.32,2491.07,2185.1000,1201.0491,0.7821,100.0,105.8489,0.1208,...,NaN,0.4974,0.0128,0.0033,2.5767,0.0223,0.0105,0.0034,47.0690,-1
1,436,2008-08-23 05:58:00,3071.58,2489.47,2217.3777,1425.1041,1.7585,100.0,106.2556,0.1200,...,16.6695,0.5004,0.0316,0.0066,6.3183,0.0329,0.0055,0.0022,16.6695,-1
2,635,2008-01-09 20:51:00,3017.53,2524.09,2201.0667,880.2317,1.4148,100.0,106.5478,0.1211,...,NaN,0.4998,0.0097,0.0026,1.9495,0.0328,0.0235,0.0068,71.5333,-1
3,996,2008-09-21 19:19:00,2901.62,2569.45,2223.9000,1745.3724,1.9974,100.0,96.7567,0.1241,...,NaN,0.5004,0.0174,0.0034,3.4771,0.0200,0.0205,0.0061,102.5241,-1
4,782,2008-11-09 07:43:00,2982.59,2466.86,2117.5889,894.0996,1.4330,100.0,106.4944,0.1253,...,NaN,0.4986,0.0172,0.0038,3.4561,0.0373,0.0079,0.0030,21.0599,-1


,id,time,0,1,2,3,4,5,6,7,...,580,581,582,583,584,585,586,587,588,589
0,246,2008-08-18 04:35:00,2985.06,2509.80,2177.3222,1089.3655,1.3101,100.0,101.1478,0.1216,...,NaN,NaN,0.4989,0.0095,0.0031,1.8958,0.0154,0.0271,0.0104,176.0329
1,1315,2008-05-10 09:47:00,2942.21,2459.02,2210.8556,1665.0062,2.4475,100.0,97.1056,0.1227,...,NaN,NaN,0.5014,0.0155,0.0034,3.0998,0.0098,0.0071,0.0022,72.1094
2,444,2008-08-24 07:12:00,3114.46,2656.84,2207.0444,1269.6078,1.7571,100.0,97.0189,0.1221,...,NaN,NaN,0.5036,0.0091,0.0029,1.8001,0.0282,0.0194,0.0059,68.7444
3,296,2008-08-19 08:07:00,2864.05,2533.75,2224.0000,1510.0797,1.5611,100.0,99.8300,0.1199,...,NaN,NaN,0.5025,0.0197,0.0044,3.9136,0.0291,0.0237,0.0071,81.3456
4,1295,2008-04-10 18:25:00,3089.26,2414.63,2209.8000,1175.5508,1.1464,100.0,112.1367,0.1227,...,0.0063,59.3775,0.5000,0.0103,0.0026,2.0700,0.0308,0.0183,0.0063,59.3775


Data types
float64    590
int64        2
object       1
Name: count, dtype: int64
Target data types:
int64
Number of unique target values:
2
First 20 target values:
[-1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1]


In [ ]:
print("Target value counts:")
print(train["target"].value_counts())

print("Target percentages:")
print(
    (train["target"].value_counts(normalize=True).sort_index()*100)
)

print("Sample submission:")
display(sample_submission.head())

Target value counts:
target
-1    1170
 1      83
Name: count, dtype: int64
Target percentages:
target
-1    93.375898
 1     6.624102
Name: proportion, dtype: float64
Sample submission:


,id,target
0,246,0.5
1,1315,0.5
2,444,0.5
3,296,0.5
4,1295,0.5


In [ ]:
print("Unique labels:", sorted(train["target"].unique()))

print("Training rows", len(train))
print("Test rows", len(test))

display(train.iloc[:5,2:22])


Unique labels: [np.int64(-1), np.int64(1)]
Training rows 1253
Test rows 314


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,3075.32,2491.07,2185.1000,1201.0491,0.7821,100.0,105.8489,0.1208,1.4002,-0.0151,-0.0006,0.9738,202.9734,0.0,3.0143,399.1464,10.2629,0.9824,192.7104,12.5949
1,3071.58,2489.47,2217.3777,1425.1041,1.7585,100.0,106.2556,0.1200,1.5270,0.0066,-0.0124,0.9598,200.2884,0.0,10.2077,405.2716,9.4565,0.9596,190.8320,12.4700
2,3017.53,2524.09,2201.0667,880.2317,1.4148,100.0,106.5478,0.1211,1.3720,-0.0005,0.0052,0.9675,198.0364,0.0,10.1995,415.3266,8.9253,0.9716,189.1111,12.4871
3,2901.62,2569.45,2223.9000,1745.3724,1.9974,100.0,96.7567,0.1241,1.5950,-0.0163,0.0061,0.9835,195.7092,0.0,8.4954,399.5080,9.2262,0.9791,186.4830,12.5237
4,2982.59,2466.86,2117.5889,894.0996,1.4330,100.0,106.4944,0.1253,1.4418,-0.0096,0.0006,0.9772,198.9694,0.0,12.3992,432.6314,9.3928,0.9806,189.5765,12.5153


In [7]:
X = train.drop(columns=["id","time","target"])

y = (train["target"] ==1).astype(int)  #true/false --> 1/0

X_test = test.drop(columns=["id","time"])

print("X shape:", X.shape)
print("y shape:", y.shape)
print("X_test shape:", X_test.shape)
print("Class counts")
print(y.value_counts().sort_index())

X shape: (1253, 590)
y shape: (1253,)
X_test shape: (314, 590)
Class counts
target
0    1170
1      83
Name: count, dtype: int64


In [8]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)

X_train shape: (1002, 590)
y_train shape: (1002,)
X_val shape: (251, 590)
y_val shape: (251,)


In [15]:
imputer = SimpleImputer(strategy="median")

#fill missing values using the median of each feature, calculated from the training split.

X_train_imputed = imputer.fit_transform(X_train)
X_val_imputed = imputer.transform(X_val)
X_test_imputed = imputer.transform(X_test)

print("Training data shape:", X_train_imputed.shape)
print("Validation data shape:", X_val_imputed.shape)
print("Test data shape:", X_test_imputed.shape)

Training data shape: (1002, 590)
Validation data shape: (251, 590)
Test data shape: (314, 590)


In [ ]:
# model = RandomForestClassifier(
#     n_estimators=200,
#     class_weight="balanced",
#     random_state=42,
#     n_jobs=-1
#   )                       #class_weight="balanced" gives more importance to the minority class during training. It doesn't guarantee better results, so we'll judge it using validation metrics.

# model.fit(X_train_imputed,y_train)

# valid_probabilities = model.predict_proba(X_val_imputed)[:,1]

# print("Validation ROC-AUC",
#       roc_auc_score(y_val,valid_probabilities)
#       )                   #measures how well the model ranks defective products above normal ones. A score of 0.5 is roughly random ranking; 1.0 is perfect ranking.

# print("Validation Average Precision",
#       average_precision_score(y_val,valid_probabilities)
#       )                   #Average Precision: summarizes precision-recall performance and is particularly useful when defective products are rare.

Validation ROC-AUC 0.7286324786324786
Validation Average Precision 0.1818495116967397


1. First, understand the entire pipeline
Here's what your code does, from sensor measurements to model evaluation.
Training data

1,002 products × 590 sensor measurements

Each product has a known label: normal (0) or defective (1)

Random Forest training

Builds 200 decision trees using samples and sensor features

class_weight="balanced" makes mistakes on the rare class count more heavily in training

Validation predictions

251 products the model did not train on

predict_proba() produces a defect probability for each product

ROC-AUC

Can it rank defective products above normal ones?


Average Precision

How well does it retrieve the rare defective products?

In [11]:
models ={

"Random Forest":RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
  ),
  "Extra Trees": ExtraTreesClassifier(
      n_estimators = 300,
      class_weight = "balanced",
      random_state=42,
      n_jobs = -1
  ),
  "Hist Gradient Boosting": HistGradientBoostingClassifier(
      max_iter = 300,
      learning_rate = 0.05,
      max_leaf_nodes = 15,
      random_state = 42
  )                       #class_weight="balanced" gives more importance to the minority class during training. It doesn't guarantee better results, so we'll judge it using validation metrics.
}

results = []

for name,candidate in models.items():
  candidate.fit(X_train_imputed, y_train)

  probabilities = candidate.predict_proba(
      X_val_imputed
  )[:,1]

  results.append({
      "Model": name,
      "ROC-AUC": roc_auc_score(y_val, probabilities),
      "Average Precision": average_precision_score(y_val, probabilities),
  })

results_df = pd.DataFrame(results).sort_values(
    "Average Precision",
    ascending=False
)

print(results_df.to_string(index=False))

                 Model  ROC-AUC  Average Precision
         Random Forest 0.738436           0.187684
Hist Gradient Boosting 0.669180           0.166886
           Extra Trees 0.652464           0.141232


                 Model  ROC-AUC  Average Precision
         Random Forest 0.738436           0.187684

          Hist Gradient Boosting 0.669180           0.166886
           
           Extra Trees 0.652464           0.141232


In [12]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_results = []

for name,model in models.items():
  for fold, (train_idx, val_idx) in enumerate(cv.split(X,y)):
    X_fold_train, X_fold_val = X.iloc[train_idx], X.iloc[val_idx]
    y_fold_train, y_fold_val = y.iloc[train_idx], y.iloc[val_idx]

    imputer = SimpleImputer(strategy="median")
    X_fold_train_imputed = imputer.fit_transform(X_fold_train)
    X_fold_val_imputed = imputer.transform(X_fold_val)

    model.fit(X_fold_train_imputed, y_fold_train)

    probabilities = model.predict_proba(X_fold_val_imputed)[:,1]

    fold_results.append({
        "Model": name,
        "Fold": fold,
        "ROC-AUC": roc_auc_score(y_fold_val, probabilities),
        "Average Precision": average_precision_score(y_fold_val, probabilities),
    })

fold_results_df = pd.DataFrame(fold_results)

print(fold_results_df.round(4).to_string(index=False))

summary =(
    fold_results_df.groupby("Model")[["ROC-AUC","Average Precision"]]
    .agg(["mean","std"])
    .round(4)
)

print("Cross Validation Summmary")
print(summary)

                 Model  Fold  ROC-AUC  Average Precision
         Random Forest     0   0.7001             0.1904
         Random Forest     1   0.7016             0.1782
         Random Forest     2   0.7895             0.2780
         Random Forest     3   0.5982             0.1085
         Random Forest     4   0.6771             0.2198
           Extra Trees     0   0.7229             0.2212
           Extra Trees     1   0.6911             0.2059
           Extra Trees     2   0.8162             0.2695
           Extra Trees     3   0.6218             0.0970
           Extra Trees     4   0.7013             0.2023
Hist Gradient Boosting     0   0.6011             0.1642
Hist Gradient Boosting     1   0.7413             0.3400
Hist Gradient Boosting     2   0.7612             0.2172
Hist Gradient Boosting     3   0.6872             0.1657
Hist Gradient Boosting     4   0.6546             0.1370
Cross Validation Summmary
                       ROC-AUC         Average Precision      

In [13]:
#Tuning ExtrasTrees

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", ExtraTreesClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ))
])

# 2. Define the hyperparameter combinations to explore.
param_distributions = {
    "model__max_features": ["sqrt", 0.3, 0.5, 0.7],
    "model__min_samples_leaf": [1, 2, 3, 5, 8],
    "model__max_depth": [None, 10, 15, 25, 40],
    "model__class_weight": [None, "balanced"],
    "model__min_samples_split": [2, 5, 10, 20],
    "model__n_estimators": [300, 500]
}


cv = StratifiedKFold(
    n_splits = 5,
    shuffle = True,
    random_state = 42
)


# 4. Search 30 randomly selected hyperparameter combinations.
search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=30,
    scoring="roc_auc",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    refit=True,
    verbose=1
)


# 5. Train and evaluate the combinations.
search.fit(X, y)

# 6. Display the results.
print("Best mean CV ROC-AUC:", search.best_score_)
print("\nBest hyperparameters:")
print(search.best_params_)

# The best pipeline has already been refitted on all of X and y.
best_model = search.best_estimator_


Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best mean CV ROC-AUC: 0.7276395173453998

Best hyperparameters:
{'model__n_estimators': 500, 'model__min_samples_split': 10, 'model__min_samples_leaf': 5, 'model__max_features': 'sqrt', 'model__max_depth': None, 'model__class_weight': None}


In [18]:
#Tuning RandomForest

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    ))
])

params = {
    "model__n_estimators": [100, 200, 300],
    "model__max_features": ["sqrt", 0.3, 0.5],
    "model__min_samples_leaf": [2, 5, 8],
    "model__class_weight": [None, "balanced"]
}

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

search = RandomizedSearchCV(
    pipeline,
    param_distributions=params,
    n_iter=15,
    scoring="roc_auc",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    refit=True,
    verbose=2
)

search.fit(X, y)

print("Mean CV ROC-AUC:", search.best_score_)
print("Best parameters:", search.best_params_)

Fitting 3 folds for each of 15 candidates, totalling 45 fits
Mean CV ROC-AUC: 0.7439096911319134
Best parameters: {'model__n_estimators': 300, 'model__min_samples_leaf': 5, 'model__max_features': 'sqrt', 'model__class_weight': None}


In [23]:
hgb_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", HistGradientBoostingClassifier(
        max_iter=150,
        learning_rate=0.05,
        max_leaf_nodes=15,
        l2_regularization=1.0,
        random_state=42
    ))
])

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    hgb_pipeline,
    X,
    y,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

print("Fold ROC-AUC:", scores)
print("Mean ROC-AUC:", scores.mean())
print("Standard deviation:", scores.std())

Fold ROC-AUC: [0.66245421 0.79423077 0.70759734]
Mean ROC-AUC: 0.7214274408718854
Standard deviation: 0.05467917932363404


In [25]:
from scipy.stats import rankdata
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import QuantileTransformer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression

# ---------- 1. Features (time + junk-column removal) ----------
def build(df):
    out = df.drop(columns=["id", "time", "target"], errors="ignore").copy()
    t = pd.to_datetime(df["time"])
    out["t_num"] = (t - pd.Timestamp("2008-01-01")).dt.total_seconds() / 86400
    out["hour"] = t.dt.hour
    out["dow"] = t.dt.dayofweek
    return out

X_all = build(train)
X_te  = build(test)

# Drop columns using only feature stats (no labels involved)
combined = pd.concat([X_all, X_te])
keep = [c for c in combined.columns
        if combined[c].nunique(dropna=True) > 1 and combined[c].isna().mean() < 0.4]
X_all, X_te = X_all[keep], X_te[keep]
print("Features kept:", len(keep))

# ---------- 2. Candidate pipelines ----------
imp = lambda: SimpleImputer(strategy="median")

pipes = {
    "ET": Pipeline([("imp", imp()),
        ("m", ExtraTreesClassifier(n_estimators=1000, min_samples_split=10,
              min_samples_leaf=5, max_features="sqrt", random_state=42, n_jobs=-1))]),
    "RF": Pipeline([("imp", imp()),
        ("m", RandomForestClassifier(n_estimators=1000, min_samples_leaf=3,
              max_features="sqrt", random_state=42, n_jobs=-1))]),
    "HGB": Pipeline([("imp", imp()),
        ("m", HistGradientBoostingClassifier(learning_rate=0.03, max_iter=200,
              max_leaf_nodes=6, min_samples_leaf=20, l2_regularization=5.0,
              max_features=0.3, random_state=42))]),
    "LR": Pipeline([("imp", imp()),
        ("q", QuantileTransformer(n_quantiles=200, output_distribution="normal")),
        ("sel", SelectKBest(f_classif, k=50)),
        ("m", LogisticRegression(C=0.05, max_iter=2000))]),
}

# ---------- 3. Repeated CV: single models + rank blend ----------
def rank01(p):
    return rankdata(p) / len(p)

seeds = [0, 1, 2]
scores = {name: [] for name in list(pipes) + ["BLEND"]}

for seed in seeds:
    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=seed
    )

    fold_scores = {name: [] for name in scores}

    for tr, va in skf.split(X_all, y):
        fold_predictions = {}

        for name, pipe in pipes.items():
            pipe.fit(X_all.iloc[tr], y.iloc[tr])

            probabilities = pipe.predict_proba(
                X_all.iloc[va]
            )[:, 1]

            # Rank predictions within this validation fold.
            fold_predictions[name] = rank01(probabilities)

            # Calculate AUC only within this fold.
            auc = roc_auc_score(
                y.iloc[va],
                fold_predictions[name]
            )
            fold_scores[name].append(auc)

        # Blend the ranked predictions for this fold.
        blend_predictions = np.mean(
            list(fold_predictions.values()),
            axis=0
        )

        fold_scores["BLEND"].append(
            roc_auc_score(y.iloc[va], blend_predictions)
        )

    # Average the five fold AUCs for each seed.
    for name in scores:
        scores[name].append(np.mean(fold_scores[name]))

for name, values in scores.items():
    print(
        f"{name:6s} mean AUC {np.mean(values):.4f} "
        f"(per-seed: {np.round(values, 3)})"
    )

# ---------- 4. Final fit on all data + submission ----------
test_preds = []
for name, p in pipes.items():
    p.fit(X_all, y)
    test_preds.append(rank01(p.predict_proba(X_te)[:, 1]))

submission = pd.DataFrame({"id": test["id"], "target": np.mean(test_preds, axis=0)})
submission.to_csv("submission.csv", index=False)
print(submission.describe())

Features kept: 445
ET     mean AUC 0.7567 (per-seed: [0.752 0.772 0.747])
RF     mean AUC 0.7328 (per-seed: [0.71  0.745 0.744])
HGB    mean AUC 0.7092 (per-seed: [0.716 0.707 0.705])
LR     mean AUC 0.6818 (per-seed: [0.69  0.692 0.663])
BLEND  mean AUC 0.7516 (per-seed: [0.744 0.765 0.746])
                id      target
count   314.000000  314.000000
mean    812.445860    0.501592
std     459.186388    0.247310
min       1.000000    0.026274
25%     436.250000    0.300358
50%     827.000000    0.486863
75%    1224.250000    0.695860
max    1565.000000    0.996019


In [26]:
submission = pd.read_csv("submission.csv")

assert list(submission.columns) == ["id", "target"]
assert len(submission) == len(test) == 314
assert submission["id"].is_unique
assert submission["target"].notna().all()
assert np.isfinite(submission["target"]).all()

print("Submission checks passed!")
print(submission.head())

Submission checks passed!
     id    target
0   246  0.716561
1  1315  0.597930
2   444  0.685510
3   296  0.441083
4  1295  0.550955


In [27]:
from sklearn.base import clone

# Use the existing Extra Trees pipeline
et_model = clone(pipes["ET"])

# Train on all labeled training data
et_model.fit(X_all, y)

# Predict defect scores for the test data
et_probabilities = et_model.predict_proba(X_te)[:, 1]

# Rank scores for ROC-AUC evaluation
et_scores = rank01(et_probabilities)

# Build submission
et_submission = pd.DataFrame({
    "id": test["id"],
    "target": et_scores
})

et_submission.to_csv("submission_et.csv", index=False)

print(et_submission.head())
print("Rows:", len(et_submission))

     id    target
0   246  0.640127
1  1315  0.703822
2   444  0.729299
3   296  0.464968
4  1295  0.659236
Rows: 314


In [28]:
from sklearn.base import clone


lr_model = clone(pipes["LR"])

# Train on all labeled training data
lr_model.fit(X_all, y)

# Predict defect scores for the test data
lr_probabilities = lr_model.predict_proba(X_te)[:, 1]

# Rank scores for ROC-AUC evaluation
lr_scores = rank01(lr_probabilities)

# Build submission
lr_submission = pd.DataFrame({
    "id": test["id"],
    "target": lr_scores
})

lr_submission.to_csv("submission_lr.csv", index=False)

print(lr_submission.head())
print("Rows:", len(lr_submission))

     id    target
0   246  0.812102
1  1315  0.624204
2   444  0.856688
3   296  0.308917
4  1295  0.347134
Rows: 314


In [23]:
# @title
test_probabilities = model.predict_proba(X_test_imputed)[:, 1]

submission = pd.DataFrame({
    "id": test["id"],
    "target": test_probabilities
})

submission.to_csv("submission.csv", index=False)

print(submission.head())
print("Submission shape:", submission.shape)
print("Probability range:",
      submission["target"].min(),
      "to",
      submission["target"].max()
      )

     id    target
0   246  0.000139
1  1315  0.000027
2   444  0.000660
3   296  0.001208
4  1295  0.000178
Submission shape: (314, 2)
Probability range: 1.5514501777738298e-07 to 0.057708881058637536


In [ ]:
# @title
print("Columns:", submission.columns.tolist())
print("Number of probabilities:", len(submission))
print("Expected test rows:", len(test))

print("Missing Values:")
print(submission.isnull().sum())

print("\nIDs match test set:",
      submission["id"].equals(test["id"]))

print("\nAll probabilities between 0 and 1:",
      submission["target"].between(0, 1).all())

print("Duplicate IDs:", submission["id"].duplicated().sum())

Columns: ['id', 'target']
Number of probabilities: 314
Expected test rows: 314
Missing Values:
id        0
target    0
dtype: int64

IDs match test set: True

All probabilities between 0 and 1: True
Duplicate IDs: 0


In [ ]:
# @title
valid_predictions = (valid_probabilities >= 0.5).astype(int)  #What does .astype(int) do? It converts Boolean values into integers:- True becomes 1. - False becomes 0.


print("Confusion Matrix:")
print(confusion_matrix(y_val, valid_predictions))

print("Classification report:")
print(classification_report(
    y_val,
    valid_predictions,
    target_names=["Normal", "Defective"]
))

Confusion Matrix:
[[234   0]
 [ 17   0]]
Classification report:
              precision    recall  f1-score   support

      Normal       0.93      1.00      0.96       234
   Defective       0.00      0.00      0.00        17

    accuracy                           0.93       251
   macro avg       0.47      0.50      0.48       251
weighted avg       0.87      0.93      0.90       251



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Here's what happened:

234 normal products correctly identified

True negatives — correct predictions.




0 normal products incorrectly flagged

False positives — none.




17 defective products missed

False negatives — every defective product was classified as normal.



Your validation set contains 251 products:
- 234 normal
- 17 defective
- Total correctly classified: 234
- Total incorrectly classified: 17

In [ ]:
# @title
# Look at predicted probabilities for actual defective products
defective_probs = valid_probabilities[y_val == 1]

print("Highest probability assigned to a defective product:",
      defective_probs.max())

print("Lowest probability assigned to a defective product:",
      defective_probs.min())

print("Defective products with probability >= 0.5:",
      (defective_probs >= 0.5).sum())

Highest probability assigned to a defective product: 0.22
Lowest probability assigned to a defective product: 0.04
Defective products with probability >= 0.5: 0


In [ ]:
# @title
# Test thresholds from 0.04 to 0.22, in steps of 0.01
thresholds = np.round(np.arange(0.04, 0.221, 0.01), 2)

results = []

for threshold in thresholds:
    predictions = (valid_probabilities >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_val, predictions, labels=[0, 1]
    ).ravel()

    precision = precision_score(
        y_val, predictions, zero_division=0
    )
    recall = recall_score(
        y_val, predictions, zero_division=0
    )
    f1 = f1_score(
        y_val, predictions, zero_division=0
    )

    results.append({
        "Threshold": threshold,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Accuracy": accuracy_score(y_val, predictions),
        "Defects_Caught": tp,
        "Defects_Missed": fn,
        "False_Alarms": fp,
        "True_Negatives": tn
    })

results_df = pd.DataFrame(results)

# Show every threshold, best F1 first
results_df = results_df.sort_values(
    by=["F1", "Recall", "Precision"],
    ascending=False
)

print(results_df.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

print("\n--- Best threshold by F1-score ---")
print(results_df.iloc[0].to_string())

 Threshold  Precision  Recall    F1  Accuracy  Defects_Caught  Defects_Missed  False_Alarms  True_Negatives
     0.120      0.182   0.353 0.240     0.849               6              11            27             207
     0.160      0.333   0.176 0.231     0.920               3              14             6             228
     0.130      0.185   0.294 0.227     0.865               5              12            22             212
     0.110      0.162   0.353 0.222     0.833               6              11            31             203
     0.150      0.300   0.176 0.222     0.916               3              14             7             227
     0.070      0.124   0.765 0.213     0.618              13               4            92             142
     0.060      0.115   0.882 0.204     0.534              15               2           115             119
     0.090      0.118   0.471 0.188     0.725               8               9            60             174
     0.080      0.110   0.52